# LangChain Payment Retry — Double Charge

## The Problem

LangChain passes tool errors back to the LLM, which decides to retry.
When a payment tool times out **after** Stripe has already charged the card —
a real and common production failure — the retry creates a second charge.

Each call is individually valid. The framework is doing exactly what it
should. **There is no `if`-condition that catches this.**

## The Fix

FiGuard's idempotency key is tied to the business operation (invoice ID +
amount), not the attempt. The first `authorize()` reserves the funds and
records the key. On retry, FiGuard returns the same `event_id` — the tool
skips Stripe and confirms the original event. One charge. Always.

In [ ]:
!pip install figuard langchain langchain-openai stripe -q

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
MODE = "simulation"  # Change to "real" to use your own API keys

if MODE == "real":
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")   # Add via Colab Secrets panel (🔑)
    STRIPE_TEST_KEY = userdata.get("STRIPE_TEST_KEY") # Only needed for langchain scenario
else:
    OPENAI_API_KEY = None
    STRIPE_TEST_KEY = None

FIGUARD_BASE_URL = "https://figuard-sandbox-1.onrender.com"
FIGUARD_API_KEY  = "sb_live_demo"

# Note: FiGuard always runs against the live sandbox — real authorization
# decisions and spend tree — even in simulation mode.

In [ ]:
# ── Display helpers ────────────────────────────────────────────────────────────
def section(title):
    print(f"\n{'═' * 64}")
    print(f"  {title}")
    print(f"{'═' * 64}")

def ok(msg):   print(f"  ✓  {msg}")
def bad(msg):  print(f"  ✗  {msg}")
def info(msg): print(f"     {msg}")
def step(msg): print(f"  →  {msg}")

In [ ]:
# ── Sandbox warmup ─────────────────────────────────────────────────────────────
# Connecting to the FiGuard sandbox warms up the Render instance.
# The budget created here is used in Part 2.
import uuid
from figuard import FiGuardClient

figuard = FiGuardClient(api_key=FIGUARD_API_KEY, base_url=FIGUARD_BASE_URL)
print("Connecting to FiGuard sandbox…", end=" ", flush=True)
budget = figuard.create_budget(
    user_id="billing_agent",
    total_limit=500.00,
    currency="USD",
    expires_in="1h",
)
print("ready.")
print(f"Budget ID : {budget.id}")
print(f"Dashboard : {FIGUARD_BASE_URL}/ui")

## Part 1 — Without FiGuard

Invoice INV-1234, $50.00. The payment tool times out *after* Stripe charges the card.
LangChain returns the error to the LLM. The LLM retries the tool.
Stripe processes the charge a second time.

In [ ]:
# ── Stripe simulator ───────────────────────────────────────────────────────────
from typing import Optional

class StripeClient:
    """Wraps Stripe in test mode, or a simulator with identical call semantics."""

    def __init__(self, mode: str, api_key: Optional[str] = None):
        self.mode = mode
        self._charges = []
        if mode == "real":
            import stripe as _stripe
            _stripe.api_key = api_key
            self._stripe = _stripe

    def charge(self, amount: float, invoice_id: str, simulate_timeout: bool = False) -> str:
        """
        Charges a card. When simulate_timeout=True, raises TimeoutError
        AFTER processing — mimicking a network drop between Stripe's
        response and our receipt of it. The charge exists in Stripe.
        We just never got the confirmation.
        """
        if self.mode == "real":
            intent = self._stripe.PaymentIntent.create(
                amount=int(amount * 100),
                currency="usd",
                payment_method="pm_card_visa",
                confirm=True,
                metadata={"invoice_id": invoice_id},
            )
            charge_id = intent.id
        else:
            charge_id = f"sim_ch_{uuid.uuid4().hex[:14]}"

        self._charges.append({"id": charge_id, "amount": amount, "invoice": invoice_id})
        print(f"     [Stripe] Charged ${amount:.2f} for {invoice_id} → {charge_id}")

        if simulate_timeout:
            raise TimeoutError(
                "Network timeout — Stripe processed the charge but the "
                "response never arrived."
            )
        return charge_id

    @property
    def charges(self): return list(self._charges)

    def reset(self): self._charges.clear()


stripe = StripeClient(mode=MODE, api_key=STRIPE_TEST_KEY)

section("PART 1 — Without FiGuard")
info("Invoice INV-1234, $50.00. Tool times out after Stripe charges.")
info("LangChain returns the error to the LLM. LLM retries the tool.")
info("Stripe processes the charge a second time.\n")

def payment_tool_unsafe(invoice_id: str, amount: float, attempt: int) -> str:
    step(f"Attempt {attempt}: charging ${amount:.2f} for {invoice_id}")
    charge_id = stripe.charge(
        amount, invoice_id,
        simulate_timeout=(attempt == 1),
    )
    return f"Payment succeeded: {charge_id}"

if MODE == "simulation":
    for attempt in [1, 2]:
        try:
            result = payment_tool_unsafe("INV-1234", 50.00, attempt)
            ok(result)
        except TimeoutError as exc:
            bad(f"Tool raised: {exc}")
            info("LangChain passes error to LLM → LLM decides to retry")
else:
    import os
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

    from langchain_openai import ChatOpenAI
    from langchain.agents import AgentExecutor, create_tool_calling_agent
    from langchain_core.tools import tool
    from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

    attempt_counter = {"n": 0}

    @tool
    def process_payment_unsafe(invoice_id: str, amount: float) -> str:
        """Process a payment for an invoice."""
        attempt_counter["n"] += 1
        return payment_tool_unsafe(invoice_id, amount, attempt_counter["n"])

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a billing agent. Charge invoices when asked. "
         "If the tool raises an error, retry once."),
        ("human", "{input}"),
        MessagesPlaceholder("agent_scratchpad"),
    ])
    agent = create_tool_calling_agent(llm, [process_payment_unsafe], prompt)
    executor = AgentExecutor(
        agent=agent, tools=[process_payment_unsafe],
        handle_tool_error=True, verbose=True,
    )
    executor.invoke({"input": "Charge $50.00 for invoice INV-1234."})

charges = stripe.charges
print(f"\n  Stripe charges: {len(charges)}")
for c in charges:
    print(f"     {c['id']}  ${c['amount']:.2f}  ({c['invoice']})")
bad(f"Customer charged ${sum(c['amount'] for c in charges):.2f} for a $50.00 invoice\n")
if MODE == "real":
    info("Check your Stripe test dashboard — both charges are there.")

## Part 2 — With FiGuard

Same scenario. The idempotency key is tied to the invoice + amount, not the attempt.
On retry, FiGuard returns the same `event_id` — the tool skips Stripe and confirms
the original event. Stripe is never called a second time.

In [ ]:
section("PART 2 — With FiGuard")
info("Same scenario. Idempotency key is tied to the invoice + amount,")
info("not the attempt. Retry finds the original authorization.")
info("Stripe is never called a second time.\n")

stripe.reset()

token = budget.session_token
ok(f"Budget: {budget.id}  (limit: $500.00)")
info("")

# Tracks which FiGuard event IDs have already been sent to Stripe.
# In production this lives in your database or a distributed cache.
# Here it proves the point: on retry, FiGuard returns the same event_id —
# that's the signal to skip Stripe.
stripe_sent = {}  # event_id → charge_id (or timeout marker)

def payment_tool_safe(invoice_id: str, amount: float, attempt: int) -> str:
    # Key is tied to the business operation — same for every attempt.
    idempotency_key = f"invoice-{invoice_id}-usd-{int(amount * 100)}"

    step(f"Attempt {attempt}: authorize ${amount:.2f} for {invoice_id}")
    auth = figuard.authorize(
        session_token=token,
        agent_id="billing_agent",
        action_type="PAYMENT",
        description=f"Payment for invoice {invoice_id}",
        requested_quantity=amount,
        idempotency_key=idempotency_key,
    )

    if not auth.is_authorized:
        raise Exception(f"Payment blocked: {auth.denial_reason}")

    ok(f"[FiGuard] Authorized — event {auth.event_id}")

    # FiGuard returns the same event_id for every retry on the same key.
    # If we already called Stripe for this event, skip it — the card
    # was already charged. Confirm the original event and return.
    if auth.event_id in stripe_sent:
        ok(f"[FiGuard] Same event returned — Stripe already charged")
        step("Skipping Stripe. Confirming original event.")
        figuard.confirm_event(auth.event_id, confirmed_quantity=amount)
        return f"Already processed — confirmed event {auth.event_id}"

    try:
        charge_id = stripe.charge(
            amount, invoice_id,
            simulate_timeout=(attempt == 1),
        )
        stripe_sent[auth.event_id] = charge_id
        figuard.confirm_event(auth.event_id, confirmed_quantity=amount)
        return f"Payment succeeded: {charge_id}"

    except TimeoutError as exc:
        # Stripe charged but we timed out before confirming.
        # Record that this event was already sent to Stripe so the
        # retry skips it. Do NOT void — the money already moved.
        stripe_sent[auth.event_id] = f"timeout_attempt_{attempt}"
        raise

if MODE == "simulation":
    for attempt in [1, 2]:
        try:
            result = payment_tool_safe("INV-1234", 50.00, attempt)
            ok(result)
        except TimeoutError as exc:
            bad(f"Tool raised: {exc}")
            info("LangChain passes error to LLM → LLM decides to retry")
else:
    attempt_counter["n"] = 0

    @tool
    def process_payment_safe(invoice_id: str, amount: float) -> str:
        """Process a payment for an invoice using FiGuard idempotency."""
        attempt_counter["n"] += 1
        return payment_tool_safe(invoice_id, amount, attempt_counter["n"])

    agent = create_tool_calling_agent(llm, [process_payment_safe], prompt)
    executor = AgentExecutor(
        agent=agent, tools=[process_payment_safe],
        handle_tool_error=True, verbose=True,
    )
    executor.invoke({"input": "Charge $50.00 for invoice INV-1234."})

charges = stripe.charges
print(f"\n  Stripe charges: {len(charges)}")
for c in charges:
    print(f"     {c['id']}  ${c['amount']:.2f}  ({c['invoice']})")
ok(f"Customer charged ${sum(c['amount'] for c in charges):.2f} — correct amount")
if MODE == "real":
    info("Check your Stripe test dashboard — one charge only.")

## What FiGuard Recorded

Open the dashboard and find the budget printed above.
Go to **Budget → Ledger** to see the event log.

In [ ]:
section("What FiGuard recorded")
info(f"Dashboard : {FIGUARD_BASE_URL}/ui")
info(f"Budget    : {budget.id}")
info("")
info("Open the budget → Ledger. You'll see:")
info("  ✓ CONFIRMED  $50.00  Payment for invoice INV-1234")
info("")
info("One event. One charge. The retry was absorbed by FiGuard's")
info("idempotency layer — same event_id returned, Stripe skipped.")
print(f"\n  Dashboard link: {FIGUARD_BASE_URL}/ui")